<a href="https://colab.research.google.com/github/r-minhyuk/deep-learning-utils/blob/main/models/skip_gram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install konlpy

!pip install Korpora

In [ ]:
#Skip-gram 모델 실습
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus=Korpora.load('nsmc')
corpus=pd.DataFrame(corpus.test)

tokenizer=Okt()
tokens=[tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [ ]:
#단어사전 구축(token_to_id 등)
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):   #special_tokens는 리스트를 받음
  counter=Counter()
  for tokens in corpus:
    counter.update(tokens)
  vocab=special_tokens

  for token, count in counter.most_common(n_vocab):
    vocab.append(token)

  return vocab



vocab=build_vocab(tokens, 5000, ['<unk>'])
tokens_to_id={token:idx for idx, token in enumerate(vocab)}
id_to_tokens={idx:token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [ ]:
#skip-gram의 단어 쌍 추출하는 함수
#[중심단어, 주변단어] 리스트로 반환
def get_word_pairs(tokens, window_size):
  pairs=[]
  for sentence in tokens:
    sentence_length=len(sentence)

    for idx, center_word in enumerate(sentence):
      window_start=max(0, idx-window_size)
      window_end=min(sentence_length, idx+window_size+1)
      context_words=sentence[window_start:idx]+sentence[idx+1:window_end]
      for context_word in context_words:
        pairs.append([center_word, context_word])

  return pairs


word_pairs=get_word_pairs(tokens, 2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [ ]:
#추출한 단어 쌍을 인덱스로 변환하는 함수
def get_index_pairs(word_pairs, tokens_to_id):
  pairs=[]
  unk_index=tokens_to_id['<unk>']
  for word_pair in word_pairs:
    center_word, context_word=word_pair
    center_index=tokens_to_id.get(center_word, unk_index)
    context_index=tokens_to_id.get(context_word, unk_index)
    pairs.append([center_index, context_index])

  return pairs


index_pairs=get_index_pairs(word_pairs, tokens_to_id)
print(index_pairs[:5])

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]


In [ ]:
#dataloader 적용
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs=torch.tensor(index_pairs)
center_indexes=index_pairs[:,0]
context_indexes=index_pairs[:,1]

dataset=TensorDataset(center_indexes, context_indexes)
dataloader=DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

In [ ]:
#embedding(skip-gram) 모델 선언
from torch import nn

class VanillaSkipgram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size, embedding_dim)
    self.linear=nn.Linear(embedding_dim, vocab_size)


  def forward(self, input_ids):
    embedding=self.embedding(input_ids)
    output=self.linear(embedding)
    return output

In [ ]:
#skip_gram 모델 학습 준비 작업
from torch import optim

device='cuda' if torch.cuda.is_available() else 'cpu'
word2vec=VanillaSkipgram(len(vocab), embedding_dim=128).to(device)
optimizer=optim.SGD(word2vec.parameters(), lr=1e-1)
criterion=nn.CrossEntropyLoss().to(device)

for epoch in range(10):
  cost=0.0
  for input_ids, output_ids in dataloader:
    input_ids=input_ids.to(device)
    output_ids=output_ids.to(device)

    logits=word2vec(input_ids)
    loss=criterion(logits, output_ids)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    cost+=loss

  cost/=len(dataloader)

  print(f'Epoch: {epoch+1:4d}, Cost: {cost:.3f}')

In [ ]:
#단어 임베딩 값 추출
token_to_embedding=dict()
embedding_matrix=word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
  token_to_embedding[word]=embedding

index=30
token=vocab[30]
token_embedding=token_to_embedding[token]

print(token)
print(token_embedding)

In [ ]:
#단어 임베딩 유사도 계산
import numpy as np
from numpy.linalg import norm

def cos_similarity(a,b):   #a에 토큰 임베딩값, b에 임베딩 행렬
  cosine=np.dot(b,a)/(norm(b, axis=1)*norm(a))
  return cosine


def top_n_index(cosine_matrix, n):
  closest_indexes=cosine_matrix.argsort[::-1]
  top_n=closest_indexes[1:n+1]
  return top_n